In [536]:
import pandas as pd
import itertools
import plotly.express as px
import plotly.graph_objects as go
import math
import numpy as np
from datetime import datetime, timedelta
import numpy as np
from scipy.stats import linregress
import matplotlib.pyplot as plt

In [273]:
cset_colors = ["#0B1F41","#003DA6","#B53A6D","#7AC4A5","#F17F4C","#15AFD0","#839DC5"]

In [248]:
#loading the raw data from epoch.ai
all_ai_models = pd.read_csv('../data/ai_models/all_ai_models.csv')

In [249]:
#split out domain list for each model, and add flag if biology is in it
all_ai_models['Domains'] = [d.split(',') if type(d) == str else None for d in all_ai_models['Domain']]
all_ai_models['is_Biology'] = all_ai_models['Domain'].str.contains('Biology') == True
all_ai_models['Model Date'] = [datetime.strptime(d,"%Y-%m-%d").date() if type(d) == str else None for d in all_ai_models['Publication date']]

In [250]:
#List all domains in the data
all_domains = set(itertools.chain.from_iterable([y for y in all_ai_models['Domains'] if y != None]))
all_domains

{'3D modeling',
 'Astronomy',
 'Audio',
 'Biology',
 'Cybersecurity',
 'Driving',
 'Earth science',
 'Games',
 'Image generation',
 'Language',
 'Materials science',
 'Mathematics',
 'Medicine',
 'Multimodal',
 'Other',
 'Psychology',
 'Recommendation',
 'Robotics',
 'Search',
 'Speech',
 'Video',
 'Vision'}

In [251]:
#getting dataset sizes
data_sizes = []
for ds in all_ai_models['Training dataset size (total)']:
    if type(ds) == str:
        sizes = [float(x) for x in ds.split(',')]
        data_sizes.append(max(sizes))
    else:
        if math.isnan(ds):
            data_sizes.append(None)
all_ai_models['Training Dataset Size'] = data_sizes

In [492]:
#getting biodata for plotting
mindate = datetime(2010,1,1).date()
plot_df = all_ai_models[all_ai_models['is_Biology'] & (all_ai_models['Model Date'] >= mindate)]
plot_df = plot_df[['Model','Model Date','Training Dataset Size']].dropna()
plot_df = plot_df[plot_df['Training Dataset Size'] > 0]
plot_df = plot_df.sort_values('Model Date', ascending=True).reset_index()[['Model','Model Date','Training Dataset Size']]

In [537]:
# Finding largest modelst
largest_models = []
for idx, row in plot_df.iterrows():
    prev_models = plot_df.loc[:idx]
    max_size = prev_models['Training Dataset Size'].quantile(0.9)
    largest_models.append(row['Training Dataset Size'] >= max_size)
plot_df['is_largest'] = largest_models

In [538]:
color_list = [cset_colors[2] if is_largest else cset_colors[1] for is_largest in plot_df['is_largest']]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x = plot_df['Model Date'], 
    y = plot_df['Training Dataset Size'],
    mode='markers',
    marker=dict(color=color_list),
    customdata = plot_df['Model'],
    hovertemplate='<b>%{customdata} (%{x})<br><b>Training Dataset Size</b>:<br>%{y:,.0f} Data Points<extra></extra>'
))
fig.update_layout(
    title='Training Dataset Size for Biological Models',
    yaxis_type="log",
    template="plotly_white",
    width = 700, height=500,
    xaxis = dict(title='Model Date'),
    yaxis = dict(title='Training Dataset Size')
)
#fig.write_html('bio_data_size.html')
#fig.write_image('bio_data_size.png')
fig.show()

In [546]:
# finding regression
mindate = plot_df['Model Date'].min()
maxdate = datetime(2028,1,1).date()
reg_data = plot_df[plot_df['is_largest']]
x = np.array([d.days for d in reg_data['Model Date'] - mindate])
x_date = reg_data['Model Date']
y = reg_data['Training Dataset Size']
y_log = np.log(y)

In [540]:
reg = linregress(x,y_log)

In [541]:
y_pred = reg.intercept + reg.slope*x

In [558]:
mse = sum((y_pred - y_log)**2)/(len(y_log)-2)
x_mean = x.mean()
xsqr_mean = ((x - x.mean())**2).mean()

In [556]:
x_plot_days = np.linspace(0,(maxdate-mindate).days,100)
x_plot = [mindate + timedelta(days=d) for d in x_plot_days]
y_plot = reg.intercept + reg.slope*x_plot_days

In [560]:
x_term = ((x_plot_days - x_mean)**2)/xsqr_mean

In [561]:
pred_err = np.sqrt(mse*(1 + 1/len(xsqr) + x_term))
pred_min = np.exp(y_plot - pred_err)
pred_max = np.exp(y_plot + pred_err)

In [605]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x = x_plot,
    y = np.exp(y_plot),
    mode='lines',
    line=dict(
        color=cset_colors[0],
        dash='dash'
    ),
    hovertemplate='%{x}<br>%{y:,.3g} Data Points<extra></extra>',
    name='Largest Model Trend'
))
fig.add_trace(go.Scatter(
    x=x_plot, 
    y=pred_min,
    line=dict(
        color='gainsboro',
        width = 0.5
    ),
    mode='lines',
    name = 'Lower Bound',
    hovertemplate='%{x}<br>%{y:,.3g} Data Points<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=x_plot,
    y=pred_max,
    fill='tonexty', # fill area between trace0 and trace1
    mode='lines', 
    line=dict(
        color='gainsboro',
        width=0.5
    ),
    name = 'Upper Bound',
    hovertemplate='%{x}<br>%{y:,.3g} Data Points<extra></extra>',
))
fig.add_trace(go.Scatter(
    x = plot_df['Model Date'], 
    y = plot_df['Training Dataset Size'],
    mode='markers',
    marker=dict(color=cset_colors[1]),
    customdata = plot_df['Model'],
    hovertemplate='<b>%{customdata} (%{x})<br><b>Training Dataset Size</b>:<br>%{y:,.0f} Data Points<extra></extra>',
    name='All Models'
))
fig.add_trace(go.Scatter(
    x = plot_df[plot_df['is_largest']]['Model Date'], 
    y = plot_df[plot_df['is_largest']]['Training Dataset Size'],
    mode='markers',
    marker=dict(color=cset_colors[2]),
    customdata = plot_df['Model'],
    hovertemplate='<b>%{customdata} (%{x})<br><b>Training Dataset Size</b>:<br>%{y:,.0f} Data Points<extra></extra>',
    name='Largest Models'
))

fig.update_layout(
    title='Training Dataset Size for Biological Models',
    yaxis_type="log",
    template="plotly_white",
    width = 700, height=500,
    xaxis = dict(title='Model Date'),
    yaxis = dict(title='Training Dataset Size'),
)
fig.write_html('bio_data_size.html')
fig.write_image('bio_data_size.png')
fig.show()